# Lab 7: Naive Bayes, Perceptron and Logistic Regression
**Student ID:** 202611022

**Course:** IT582 – Foundations of Machine Learning

**Instructor:** Srimanta Mandal

---
## Task 1: Naive Bayes Classification on Titanic Data

Use the Titanic passenger dataset (OpenML data id=40945) with categorical attributes **pclass**, **sex**, **embarked** to predict **survived**. Implement a categorical Naive Bayes classifier from scratch with Laplace smoothing.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score
from sklearn.datasets import load_iris, fetch_openml

plt.rcParams['figure.dpi'] = 100
np.random.seed(42)

In [ ]:
# Load the Titanic dataset from OpenML
titanic = fetch_openml(data_id=40945, as_frame=True)
df = titanic.frame
print("Dataset shape:", df.shape)
print("\nColumn names:", df.columns.tolist())
print("\nFirst 5 rows:")
df.head()

In [ ]:
# Select required columns and remove missing values
features = ['pclass', 'sex', 'embarked']
target = 'survived'

df_selected = df[features + [target]].copy()
print(f"Shape before dropping NaN: {df_selected.shape}")
print(f"Missing values per column:\n{df_selected.isnull().sum()}\n")

df_selected = df_selected.dropna()
print(f"Shape after dropping NaN: {df_selected.shape}")

X = df_selected[features].values
y = df_selected[target].values

# Stratified 70-30 train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
print(f"\nTraining set size: {X_train.shape[0]}")
print(f"Test set size:     {X_test.shape[0]}")
print(f"Training class distribution: {dict(zip(*np.unique(y_train, return_counts=True)))}")
print(f"Test class distribution:     {dict(zip(*np.unique(y_test, return_counts=True)))}")

In [ ]:
# Exploratory visualization of features vs survival
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for i, feat in enumerate(features):
    df_plot = df_selected.copy()
    ct = pd.crosstab(df_plot[feat], df_plot[target], normalize='index') * 100
    ct.plot(kind='bar', stacked=True, ax=axes[i], colormap='RdYlGn')
    axes[i].set_title(f'Survival Rate by {feat}', fontsize=13)
    axes[i].set_ylabel('Percentage')
    axes[i].set_xlabel(feat)
    axes[i].legend(title='Survived', loc='upper right')
    axes[i].tick_params(axis='x', rotation=0)

plt.suptitle('Feature-wise Survival Distribution (Titanic)', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
class CategoricalNaiveBayes:
    """
    Categorical Naive Bayes classifier from scratch with Laplace smoothing.
    
    For prediction, uses log-score:
        S(c) = log P(Y=c) + sum_j log P(X_j = x_j | Y=c)
    and assigns the class with the largest score.
    """
    
    def __init__(self, alpha=1.0):
        """alpha: Laplace smoothing parameter."""
        self.alpha = alpha
        self.classes_ = None
        self.class_priors_ = None
        self.cond_probs_ = None  # list of dicts, one per feature
    
    def fit(self, X, y):
        self.classes_ = np.unique(y)
        n_samples = len(y)
        n_features = X.shape[1]
        
        # Estimate class priors: P(Y = c)
        self.class_priors_ = {}
        for c in self.classes_:
            self.class_priors_[c] = np.sum(y == c) / n_samples
        
        # Estimate conditional probabilities: P(X_j = x_j | Y = c) with Laplace smoothing
        self.cond_probs_ = []
        self.feature_values_ = []
        
        for j in range(n_features):
            unique_vals = np.unique(X[:, j])
            self.feature_values_.append(unique_vals)
            k = len(unique_vals)  # number of unique categories for this feature
            
            prob_table = {}  # {class: {feature_value: probability}}
            for c in self.classes_:
                mask = (y == c)
                n_c = np.sum(mask)
                prob_table[c] = {}
                for val in unique_vals:
                    count = np.sum(X[mask, j] == val)
                    # Laplace smoothing
                    prob_table[c][val] = (count + self.alpha) / (n_c + self.alpha * k)
            
            self.cond_probs_.append(prob_table)
        
        return self
    
    def predict_log_scores(self, X):
        """Compute log-scores for each class for all samples."""
        n_samples = X.shape[0]
        scores = np.zeros((n_samples, len(self.classes_)))
        
        for idx, c in enumerate(self.classes_):
            log_prior = np.log(self.class_priors_[c])
            
            for i in range(n_samples):
                score = log_prior
                for j in range(X.shape[1]):
                    val = X[i, j]
                    if val in self.cond_probs_[j][c]:
                        score += np.log(self.cond_probs_[j][c][val])
                    else:
                        # Handle unseen categories with Laplace smoothing
                        k = len(self.feature_values_[j])
                        n_c = np.sum(self.class_priors_[c])
                        score += np.log(self.alpha / (n_c + self.alpha * (k + 1)))
                scores[i, idx] = score
        
        return scores
    
    def predict(self, X):
        scores = self.predict_log_scores(X)
        return self.classes_[np.argmax(scores, axis=1)]

# Train the Naive Bayes classifier
nb_clf = CategoricalNaiveBayes(alpha=1.0)
nb_clf.fit(X_train, y_train)
print("Naive Bayes classifier trained successfully.")

In [ ]:
# Report estimated class priors
print("="*50)
print("ESTIMATED CLASS PRIORS")
print("="*50)
for c, prior in nb_clf.class_priors_.items():
    print(f"  P(survived = {c}) = {prior:.4f}")
print()

In [ ]:
# Display conditional probability tables
print("="*60)
print("CONDITIONAL PROBABILITY TABLES (with Laplace smoothing)")
print("="*60)

for j, feat in enumerate(features):
    print(f"\n--- Feature: {feat} ---")
    header = f"{'Value':<12}"
    for c in nb_clf.classes_:
        header += f"P({feat}=val | survived={c})  "
    print(header)
    print("-" * len(header))
    
    for val in nb_clf.feature_values_[j]:
        row = f"{str(val):<12}"
        for c in nb_clf.classes_:
            prob = nb_clf.cond_probs_[j][c][val]
            row += f"{prob:.4f}                       "
        print(row)

In [ ]:
# Visualize conditional probability tables as heatmaps
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for j, feat in enumerate(features):
    vals = nb_clf.feature_values_[j]
    data = []
    for val in vals:
        row = []
        for c in nb_clf.classes_:
            row.append(nb_clf.cond_probs_[j][c][val])
        data.append(row)
    
    df_heatmap = pd.DataFrame(data, index=vals, columns=[f'survived={c}' for c in nb_clf.classes_])
    sns.heatmap(df_heatmap, annot=True, fmt='.3f', cmap='YlOrRd', ax=axes[j], 
                linewidths=1, linecolor='gray')
    axes[j].set_title(f'P({feat} | survived)', fontsize=12)
    axes[j].set_ylabel(feat)

plt.suptitle('Conditional Probability Tables (Heatmaps)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Predict on test set
y_pred_nb = nb_clf.predict(X_test)

# Test accuracy
test_acc_nb = np.mean(y_pred_nb == y_test) * 100
print(f"Test Accuracy: {test_acc_nb:.2f}%")

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_nb, labels=nb_clf.classes_)
print(f"\nConfusion Matrix:")
print(cm)

# Plot confusion matrix
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=nb_clf.classes_, yticklabels=nb_clf.classes_,
            linewidths=1, linecolor='gray', ax=ax)
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('Actual', fontsize=12)
ax.set_title('Confusion Matrix – Naive Bayes (Titanic)', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Select 5 test passengers and display detailed predictions
print("="*90)
print("DETAILED PREDICTIONS FOR 5 TEST PASSENGERS")
print("="*90)

log_scores = nb_clf.predict_log_scores(X_test)

indices = [0, 1, 2, 3, 4]
for i in indices:
    print(f"\nPassenger {i+1}:")
    print(f"  Attributes: pclass={X_test[i,0]}, sex={X_test[i,1]}, embarked={X_test[i,2]}")
    print(f"  True class (survived): {y_test[i]}")
    for idx, c in enumerate(nb_clf.classes_):
        print(f"  Log-score for survived={c}: {log_scores[i, idx]:.4f}")
    pred = nb_clf.classes_[np.argmax(log_scores[i])]
    print(f"  Predicted class: {pred}")
    print(f"  Correct: {'✓' if pred == y_test[i] else '✗'}")

### Observations – Task 1 (Naive Bayes on Titanic)

* **Class Priors:** The estimated class priors reflect the known historical fact about the Titanic disaster — more passengers did not survive than survived. The prior for `survived=0` is notably higher than for `survived=1`, capturing the base rate of mortality.

* **Conditional Probabilities:** The conditional probability tables reveal important patterns:
  - **Sex** is the strongest predictor: females have a much higher conditional probability given survival (`survived=1`), consistent with the "women and children first" evacuation policy.
  - **Pclass** also shows clear differentiation: 1st class passengers have higher survival probabilities compared to 3rd class.
  - **Embarked** shows some variation (e.g., passengers who embarked at Cherbourg had slightly higher survival rates), though its predictive power is weaker than sex and pclass.

* **Laplace Smoothing:** By adding a pseudocount of 1 to every category count, Laplace smoothing prevents zero probabilities that would otherwise dominate the log-score calculation. This is critical when a feature-value/class combination has zero training examples.

* **Log-Score Prediction:** Using the log-score formulation `S(c) = log P(Y=c) + Σ log P(Xj=xj|Y=c)` avoids numerical underflow that would occur when multiplying many small probabilities together.

* **Test Accuracy and Confusion Matrix:** The classifier achieves reasonable accuracy using only three categorical features. The confusion matrix shows that the model tends to predict the majority class (not survived) more often, reflecting the imbalanced prior. Most errors come from predicting death for passengers who actually survived, particularly males in lower classes.

* **Naive Bayes Assumption:** The "naive" conditional independence assumption — that pclass, sex, and embarked are independent given survival — is clearly violated in reality (e.g., 1st class passengers were more likely to be female). Despite this, the classifier still performs reasonably well, demonstrating that Naive Bayes is robust to moderate violations of its assumptions.

---
## Task 2: Classification of Linearly Separable Data

Use the Iris dataset: retain only **Iris Setosa** and **Iris Versicolor** with features **Petal Length** ($x_1$) and **Petal Width** ($x_2$). Train a **Perceptron** and **Logistic Regression** from scratch.

In [ ]:
# Load Iris dataset and filter for Setosa (0) and Versicolor (1)
iris = load_iris()
X_iris = iris.data
y_iris = iris.target

# Keep only classes 0 (Setosa) and 1 (Versicolor)
mask_task2 = (y_iris == 0) | (y_iris == 1)
X_task2 = X_iris[mask_task2][:, [2, 3]]  # Petal Length (col 2), Petal Width (col 3)
y_task2 = y_iris[mask_task2]

print(f"Dataset shape: {X_task2.shape}")
print(f"Classes: {np.unique(y_task2)} -> Setosa (0), Versicolor (1)")
print(f"Class distribution: {dict(zip(*np.unique(y_task2, return_counts=True)))}")

In [ ]:
# Plot the two classes before training
fig, ax = plt.subplots(figsize=(8, 6))

colors = ['#e74c3c', '#2ecc71']
labels = ['Iris Setosa', 'Iris Versicolor']

for cls in [0, 1]:
    mask = y_task2 == cls
    ax.scatter(X_task2[mask, 0], X_task2[mask, 1], c=colors[cls], 
               label=labels[cls], edgecolors='k', alpha=0.8, s=60)

ax.set_xlabel('Petal Length (x₁)', fontsize=12)
ax.set_ylabel('Petal Width (x₂)', fontsize=12)
ax.set_title('Task 2: Setosa vs Versicolor (Linearly Separable)', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print("Observation: The two classes are clearly linearly separable with no overlap.")

In [ ]:
# Stratified 70-30 split and standardize using training-set statistics only
X_train_t2, X_test_t2, y_train_t2, y_test_t2 = train_test_split(
    X_task2, y_task2, test_size=0.3, random_state=42, stratify=y_task2
)

# Standardize features using training-set mean and std
mean_t2 = X_train_t2.mean(axis=0)
std_t2 = X_train_t2.std(axis=0)

X_train_t2_s = (X_train_t2 - mean_t2) / std_t2
X_test_t2_s = (X_test_t2 - mean_t2) / std_t2

print(f"Training set: {X_train_t2_s.shape[0]} samples")
print(f"Test set:     {X_test_t2_s.shape[0]} samples")
print(f"Training mean (after standardization): {X_train_t2_s.mean(axis=0)}")
print(f"Training std  (after standardization): {X_train_t2_s.std(axis=0)}")

### Task 2A: Perceptron

In [ ]:
class Perceptron:
    """
    Perceptron classifier from scratch.
    Labels: y ∈ {-1, +1}
    Update rule for misclassified observation: θ ← θ + α * y(i) * x(i)
    Bias is included by augmenting x with a 1.
    """
    
    def __init__(self, lr=0.1, max_epochs=100):
        self.lr = lr
        self.max_epochs = max_epochs
        self.theta = None
        self.misclassified_per_epoch = []
    
    def fit(self, X, y):
        """y should be in {-1, +1}."""
        n_samples, n_features = X.shape
        # Augment X with bias column
        X_aug = np.hstack([np.ones((n_samples, 1)), X])
        
        # Initialize weights to zeros
        self.theta = np.zeros(n_features + 1)
        self.misclassified_per_epoch = []
        
        for epoch in range(self.max_epochs):
            misclassified = 0
            for i in range(n_samples):
                prediction = np.sign(np.dot(self.theta, X_aug[i]))
                if prediction == 0:
                    prediction = -1  # treat 0 as misclassified
                
                if prediction != y[i]:
                    self.theta += self.lr * y[i] * X_aug[i]
                    misclassified += 1
            
            self.misclassified_per_epoch.append(misclassified)
            
            # Early stopping if no misclassifications
            if misclassified == 0:
                print(f"Converged at epoch {epoch + 1} (zero misclassifications).")
                break
        
        return self
    
    def predict(self, X):
        X_aug = np.hstack([np.ones((X.shape[0], 1)), X])
        predictions = np.sign(np.dot(X_aug, self.theta))
        predictions[predictions == 0] = -1
        return predictions
    
    def accuracy(self, X, y):
        return np.mean(self.predict(X) == y) * 100

In [ ]:
# Convert labels to {-1, +1} for Perceptron
y_train_t2_perc = np.where(y_train_t2 == 0, -1, 1)
y_test_t2_perc = np.where(y_test_t2 == 0, -1, 1)

# Train Perceptron
perceptron_t2 = Perceptron(lr=0.1, max_epochs=100)
perceptron_t2.fit(X_train_t2_s, y_train_t2_perc)

# Report results
train_acc_perc_t2 = perceptron_t2.accuracy(X_train_t2_s, y_train_t2_perc)
test_acc_perc_t2 = perceptron_t2.accuracy(X_test_t2_s, y_test_t2_perc)

print(f"\nNumber of epochs for convergence: {len(perceptron_t2.misclassified_per_epoch)}")
print(f"Training accuracy: {train_acc_perc_t2:.2f}%")
print(f"Test accuracy:     {test_acc_perc_t2:.2f}%")
print(f"Final weights (θ): {perceptron_t2.theta}")

In [ ]:
# Plot misclassified training samples per epoch
fig, ax = plt.subplots(figsize=(8, 5))
epochs_range = range(1, len(perceptron_t2.misclassified_per_epoch) + 1)
ax.plot(epochs_range, perceptron_t2.misclassified_per_epoch, 'o-', 
        color='#e74c3c', linewidth=2, markersize=6)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Misclassified Training Samples', fontsize=12)
ax.set_title('Perceptron: Misclassifications per Epoch (Task 2 – Linearly Separable)', fontsize=13)
ax.grid(True, alpha=0.3)
ax.set_xticks(list(epochs_range))
plt.tight_layout()
plt.show()

In [ ]:
def plot_decision_boundary_perceptron(X, y, theta, title, labels=None, ax=None):
    """Plot decision boundary for Perceptron (y in {-1,+1})."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 6))
    
    colors = ['#e74c3c', '#2ecc71']
    classes = np.unique(y)
    if labels is None:
        labels = [f'Class {c}' for c in classes]
    
    for i, cls in enumerate(classes):
        mask = y == cls
        ax.scatter(X[mask, 0], X[mask, 1], c=colors[i], 
                   label=labels[i], edgecolors='k', alpha=0.8, s=60)
    
    # Decision boundary: theta[0] + theta[1]*x1 + theta[2]*x2 = 0
    x1_range = np.linspace(X[:, 0].min() - 0.5, X[:, 0].max() + 0.5, 200)
    if abs(theta[2]) > 1e-10:
        x2_boundary = -(theta[0] + theta[1] * x1_range) / theta[2]
        ax.plot(x1_range, x2_boundary, 'k--', linewidth=2, label='Decision Boundary')
    
    ax.set_xlabel('Petal Length (standardized)', fontsize=12)
    ax.set_ylabel('Petal Width (standardized)', fontsize=12)
    ax.set_title(title, fontsize=13)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    
    return ax

# Plot Perceptron decision boundary for Task 2
fig, ax = plt.subplots(figsize=(8, 6))
plot_decision_boundary_perceptron(
    X_train_t2_s, y_train_t2_perc, perceptron_t2.theta,
    'Perceptron Decision Boundary (Task 2 – Setosa vs Versicolor)',
    labels=['Setosa (-1)', 'Versicolor (+1)'], ax=ax
)
plt.tight_layout()
plt.show()

### Task 2B: Logistic Regression

In [ ]:
class LogisticRegression:
    """
    Logistic Regression from scratch using Batch Gradient Descent.
    
    Hypothesis: h_θ(x) = 1 / (1 + exp(-θᵀx))
    Loss: Negative log-likelihood (Binary Cross-Entropy)
    Labels: y ∈ {0, 1}
    Classification threshold: 0.5
    """
    
    def __init__(self, lr=0.1, max_iter=1000):
        self.lr = lr
        self.max_iter = max_iter
        self.theta = None
        self.loss_history = []
    
    def sigmoid(self, z):
        z = np.clip(z, -500, 500)
        return 1.0 / (1.0 + np.exp(-z))
    
    def compute_loss(self, X_aug, y):
        """Compute negative log-likelihood."""
        m = len(y)
        h = self.sigmoid(X_aug @ self.theta)
        h = np.clip(h, 1e-10, 1 - 1e-10)
        loss = -(1/m) * np.sum(y * np.log(h) + (1 - y) * np.log(1 - h))
        return loss
    
    def fit(self, X, y):
        n_samples, n_features = X.shape
        X_aug = np.hstack([np.ones((n_samples, 1)), X])
        
        # Initialize weights to zeros
        self.theta = np.zeros(n_features + 1)
        self.loss_history = []
        
        for i in range(self.max_iter):
            h = self.sigmoid(X_aug @ self.theta)
            gradient = (1/n_samples) * (X_aug.T @ (h - y))
            self.theta -= self.lr * gradient
            
            loss = self.compute_loss(X_aug, y)
            self.loss_history.append(loss)
        
        return self
    
    def predict_proba(self, X):
        X_aug = np.hstack([np.ones((X.shape[0], 1)), X])
        return self.sigmoid(X_aug @ self.theta)
    
    def predict(self, X):
        return (self.predict_proba(X) >= 0.5).astype(int)
    
    def accuracy(self, X, y):
        return np.mean(self.predict(X) == y) * 100

In [ ]:
# Labels {0, 1} for Logistic Regression (already in this format)
y_train_t2_lr = y_train_t2.copy()
y_test_t2_lr = y_test_t2.copy()

# Train Logistic Regression
lr_t2 = LogisticRegression(lr=0.5, max_iter=500)
lr_t2.fit(X_train_t2_s, y_train_t2_lr)

train_acc_lr_t2 = lr_t2.accuracy(X_train_t2_s, y_train_t2_lr)
test_acc_lr_t2 = lr_t2.accuracy(X_test_t2_s, y_test_t2_lr)

print(f"Training accuracy: {train_acc_lr_t2:.2f}%")
print(f"Test accuracy:     {test_acc_lr_t2:.2f}%")
print(f"Final weights (θ): {lr_t2.theta}")
print(f"Final loss: {lr_t2.loss_history[-1]:.6f}")

In [ ]:
# Plot loss versus iteration
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(range(1, len(lr_t2.loss_history) + 1), lr_t2.loss_history, 
        color='#3498db', linewidth=2)
ax.set_xlabel('Iteration', fontsize=12)
ax.set_ylabel('Loss (Negative Log-Likelihood)', fontsize=12)
ax.set_title('Logistic Regression: Loss vs Iteration (Task 2)', fontsize=13)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
def plot_decision_boundary_lr(X, y, theta, title, labels=None, ax=None):
    """Plot decision boundary for Logistic Regression (y in {0,1})."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 6))
    
    colors = ['#e74c3c', '#2ecc71']
    classes = np.unique(y)
    if labels is None:
        labels = [f'Class {c}' for c in classes]
    
    for i, cls in enumerate(classes):
        mask = y == cls
        ax.scatter(X[mask, 0], X[mask, 1], c=colors[i], 
                   label=labels[i], edgecolors='k', alpha=0.8, s=60)
    
    # Decision boundary: theta[0] + theta[1]*x1 + theta[2]*x2 = 0
    x1_range = np.linspace(X[:, 0].min() - 0.5, X[:, 0].max() + 0.5, 200)
    if abs(theta[2]) > 1e-10:
        x2_boundary = -(theta[0] + theta[1] * x1_range) / theta[2]
        ax.plot(x1_range, x2_boundary, 'k--', linewidth=2, label='Decision Boundary')
    
    ax.set_xlabel('Petal Length (standardized)', fontsize=12)
    ax.set_ylabel('Petal Width (standardized)', fontsize=12)
    ax.set_title(title, fontsize=13)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    
    return ax

# Plot Logistic Regression decision boundary
fig, ax = plt.subplots(figsize=(8, 6))
plot_decision_boundary_lr(
    X_train_t2_s, y_train_t2_lr, lr_t2.theta,
    'Logistic Regression Decision Boundary (Task 2 – Setosa vs Versicolor)',
    labels=['Setosa (0)', 'Versicolor (1)'], ax=ax
)
plt.tight_layout()
plt.show()

In [ ]:
# Side-by-side comparison of Perceptron and Logistic Regression boundaries
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

plot_decision_boundary_perceptron(
    X_train_t2_s, y_train_t2_perc, perceptron_t2.theta,
    'Perceptron',
    labels=['Setosa (-1)', 'Versicolor (+1)'], ax=axes[0]
)

plot_decision_boundary_lr(
    X_train_t2_s, y_train_t2_lr, lr_t2.theta,
    'Logistic Regression',
    labels=['Setosa (0)', 'Versicolor (1)'], ax=axes[1]
)

plt.suptitle('Task 2: Decision Boundary Comparison (Linearly Separable Data)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### Observations – Task 2 (Perceptron vs Logistic Regression on Linearly Separable Data)

* **Perceptron Convergence:** The Perceptron converges quickly (within very few epochs) on this linearly separable data. According to the Perceptron Convergence Theorem, the algorithm is guaranteed to find a separating hyperplane in a finite number of steps when the data is linearly separable.

* **Training & Test Accuracy:** Both classifiers achieve 100% training and test accuracy, which is expected since Setosa and Versicolor are well-separated in the Petal Length–Petal Width feature space.

* **Decision Boundaries:** Both classifiers find valid separating hyperplanes, but they may differ because:
  - The **Perceptron** finds *any* separating hyperplane — its final boundary depends on the order of data points and when convergence occurred. It does not maximize margin.
  - **Logistic Regression** finds the boundary that maximizes the likelihood of the training data under the logistic model. It tends to place the boundary where the predicted probabilities transition smoothly from 0 to 1, which often appears more centered between the classes.

* **Loss Curve (Logistic Regression):** The loss decreases monotonically and converges to a near-zero value, indicating that the optimizer successfully minimizes the negative log-likelihood. The rapid initial decrease reflects the clear separability of the data.

* **Comparison:** For linearly separable data, both classifiers perform equally well in terms of accuracy. However, Logistic Regression is preferred in practice because it provides probabilistic outputs (class probabilities), which are more informative than the hard decisions of the Perceptron.

---
## Task 3: Classification of Non-Linearly Separable Data

Use the Iris dataset: retain only **Iris Versicolor** and **Iris Virginica** with features **Petal Length** ($x_1$) and **Petal Width** ($x_2$). This dataset has significant class overlap.

In [ ]:
# Filter for Versicolor (1) and Virginica (2)
mask_task3 = (y_iris == 1) | (y_iris == 2)
X_task3 = X_iris[mask_task3][:, [2, 3]]  # Petal Length, Petal Width
y_task3 = y_iris[mask_task3]

# Remap labels: Versicolor -> 0, Virginica -> 1
y_task3_binary = (y_task3 - 1).copy()  # 1->0, 2->1

print(f"Dataset shape: {X_task3.shape}")
print(f"Classes: Versicolor (0), Virginica (1)")
print(f"Class distribution: {dict(zip(*np.unique(y_task3_binary, return_counts=True)))}")

In [ ]:
# Plot the data and compare with Task 2
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Task 2 data (Setosa vs Versicolor)
for cls in [0, 1]:
    mask = y_task2 == cls
    axes[0].scatter(X_task2[mask, 0], X_task2[mask, 1], c=['#e74c3c', '#2ecc71'][cls], 
                    label=['Setosa', 'Versicolor'][cls], edgecolors='k', alpha=0.8, s=60)
axes[0].set_xlabel('Petal Length', fontsize=12)
axes[0].set_ylabel('Petal Width', fontsize=12)
axes[0].set_title('Task 2: Setosa vs Versicolor (Linearly Separable)', fontsize=13)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Task 3 data (Versicolor vs Virginica)
for cls in [0, 1]:
    mask = y_task3_binary == cls
    axes[1].scatter(X_task3[mask, 0], X_task3[mask, 1], c=['#3498db', '#e67e22'][cls], 
                    label=['Versicolor', 'Virginica'][cls], edgecolors='k', alpha=0.8, s=60)
axes[1].set_xlabel('Petal Length', fontsize=12)
axes[1].set_ylabel('Petal Width', fontsize=12)
axes[1].set_title('Task 3: Versicolor vs Virginica (Not Linearly Separable)', fontsize=13)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.suptitle('Comparison of Class Separation: Task 2 vs Task 3', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("Observation: Task 3 data (Versicolor vs Virginica) shows significant class overlap,")
print("unlike Task 2 data which is clearly linearly separable.")

In [ ]:
# Stratified 70-30 split and standardize
X_train_t3, X_test_t3, y_train_t3, y_test_t3 = train_test_split(
    X_task3, y_task3_binary, test_size=0.3, random_state=42, stratify=y_task3_binary
)

# Standardize
mean_t3 = X_train_t3.mean(axis=0)
std_t3 = X_train_t3.std(axis=0)

X_train_t3_s = (X_train_t3 - mean_t3) / std_t3
X_test_t3_s = (X_test_t3 - mean_t3) / std_t3

print(f"Training set: {X_train_t3_s.shape[0]} samples")
print(f"Test set:     {X_test_t3_s.shape[0]} samples")

### Task 3A: Linear Classifiers on Non-Linearly Separable Data

In [ ]:
# Perceptron on Task 3
y_train_t3_perc = np.where(y_train_t3 == 0, -1, 1)
y_test_t3_perc = np.where(y_test_t3 == 0, -1, 1)

perceptron_t3 = Perceptron(lr=0.1, max_epochs=100)
perceptron_t3.fit(X_train_t3_s, y_train_t3_perc)

train_acc_perc_t3 = perceptron_t3.accuracy(X_train_t3_s, y_train_t3_perc)
test_acc_perc_t3 = perceptron_t3.accuracy(X_test_t3_s, y_test_t3_perc)

print(f"Perceptron on Task 3 (Non-Linearly Separable):")
print(f"  Number of epochs run: {len(perceptron_t3.misclassified_per_epoch)}")
print(f"  Training accuracy: {train_acc_perc_t3:.2f}%")
print(f"  Test accuracy:     {test_acc_perc_t3:.2f}%")
print(f"  Final misclassifications: {perceptron_t3.misclassified_per_epoch[-1]}")

In [ ]:
# Plot misclassified vs epoch for Perceptron on Task 3
fig, ax = plt.subplots(figsize=(10, 5))
epochs_range_t3 = range(1, len(perceptron_t3.misclassified_per_epoch) + 1)
ax.plot(epochs_range_t3, perceptron_t3.misclassified_per_epoch, '-', 
        color='#e74c3c', linewidth=1.5, alpha=0.8)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Misclassified Training Samples', fontsize=12)
ax.set_title('Perceptron: Misclassifications per Epoch (Task 3 – Non-Linearly Separable)', fontsize=13)
ax.grid(True, alpha=0.3)
ax.axhline(y=0, color='green', linestyle='--', alpha=0.5, label='Zero misclassifications')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

print(f"Note: Misclassification count does NOT reach zero because the data is not linearly separable.")
print(f"The Perceptron oscillates and runs for all {len(perceptron_t3.misclassified_per_epoch)} epochs.")

In [ ]:
# Logistic Regression on Task 3
lr_t3 = LogisticRegression(lr=0.5, max_iter=500)
lr_t3.fit(X_train_t3_s, y_train_t3)

train_acc_lr_t3 = lr_t3.accuracy(X_train_t3_s, y_train_t3)
test_acc_lr_t3 = lr_t3.accuracy(X_test_t3_s, y_test_t3)

print(f"Logistic Regression on Task 3 (Non-Linearly Separable):")
print(f"  Training accuracy: {train_acc_lr_t3:.2f}%")
print(f"  Test accuracy:     {test_acc_lr_t3:.2f}%")
print(f"  Final loss: {lr_t3.loss_history[-1]:.6f}")

In [ ]:
# Plot loss vs iteration for LR on Task 3
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(range(1, len(lr_t3.loss_history) + 1), lr_t3.loss_history, 
        color='#3498db', linewidth=2)
ax.set_xlabel('Iteration', fontsize=12)
ax.set_ylabel('Loss (Negative Log-Likelihood)', fontsize=12)
ax.set_title('Logistic Regression: Loss vs Iteration (Task 3)', fontsize=13)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Decision boundaries for both classifiers on Task 3
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

plot_decision_boundary_perceptron(
    X_train_t3_s, y_train_t3_perc, perceptron_t3.theta,
    'Perceptron (Task 3)',
    labels=['Versicolor (-1)', 'Virginica (+1)'], ax=axes[0]
)

plot_decision_boundary_lr(
    X_train_t3_s, y_train_t3, lr_t3.theta,
    'Logistic Regression (Task 3)',
    labels=['Versicolor (0)', 'Virginica (1)'], ax=axes[1]
)

plt.suptitle('Task 3: Decision Boundaries on Non-Linearly Separable Data', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### Task 3B: Nonlinear Feature Expansion for Logistic Regression

In [ ]:
def expand_features(X):
    """
    Nonlinear feature expansion:
    φ(x) = [1, x1, x2, x1*x2, x1², x2²]ᵀ
    
    Note: The bias (1) will be added by the LogisticRegression class,
    so here we return [x1, x2, x1*x2, x1², x2²].
    """
    x1 = X[:, 0]
    x2 = X[:, 1]
    return np.column_stack([x1, x2, x1 * x2, x1**2, x2**2])

# Expand features
X_train_t3_expanded = expand_features(X_train_t3_s)
X_test_t3_expanded = expand_features(X_test_t3_s)

print(f"Original feature shape:  {X_train_t3_s.shape}")
print(f"Expanded feature shape:  {X_train_t3_expanded.shape}")
print(f"Features: [x1, x2, x1*x2, x1², x2²]")

In [ ]:
# Train Logistic Regression with expanded features
lr_t3_expanded = LogisticRegression(lr=0.1, max_iter=1000)
lr_t3_expanded.fit(X_train_t3_expanded, y_train_t3)

train_acc_lr_t3_exp = lr_t3_expanded.accuracy(X_train_t3_expanded, y_train_t3)
test_acc_lr_t3_exp = lr_t3_expanded.accuracy(X_test_t3_expanded, y_test_t3)

print(f"Logistic Regression with Nonlinear Feature Expansion (Task 3):")
print(f"  Training accuracy: {train_acc_lr_t3_exp:.2f}%")
print(f"  Test accuracy:     {test_acc_lr_t3_exp:.2f}%")
print(f"  Final loss: {lr_t3_expanded.loss_history[-1]:.6f}")

print(f"\nComparison of Test Accuracies:")
print(f"  Linear LR:   {test_acc_lr_t3:.2f}%")
print(f"  Expanded LR: {test_acc_lr_t3_exp:.2f}%")

In [ ]:
# Plot nonlinear decision boundary in original (x1, x2) plane
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Linear LR decision boundary
plot_decision_boundary_lr(
    X_train_t3_s, y_train_t3, lr_t3.theta,
    'Linear Logistic Regression',
    labels=['Versicolor (0)', 'Virginica (1)'], ax=axes[0]
)

# Nonlinear LR decision boundary
# Create a mesh grid in the original (x1, x2) space
x1_min, x1_max = X_train_t3_s[:, 0].min() - 1, X_train_t3_s[:, 0].max() + 1
x2_min, x2_max = X_train_t3_s[:, 1].min() - 1, X_train_t3_s[:, 1].max() + 1
xx1, xx2 = np.meshgrid(np.linspace(x1_min, x1_max, 300),
                        np.linspace(x2_min, x2_max, 300))

grid_points = np.column_stack([xx1.ravel(), xx2.ravel()])
grid_expanded = expand_features(grid_points)
Z = lr_t3_expanded.predict(grid_expanded).reshape(xx1.shape)

# Plot
axes[1].contourf(xx1, xx2, Z, alpha=0.3, cmap='RdYlBu', levels=1)
axes[1].contour(xx1, xx2, Z, colors='k', linewidths=2, levels=[0.5])

colors = ['#3498db', '#e67e22']
labels_nl = ['Versicolor (0)', 'Virginica (1)']
for cls in [0, 1]:
    mask = y_train_t3 == cls
    axes[1].scatter(X_train_t3_s[mask, 0], X_train_t3_s[mask, 1], c=colors[cls], 
                    label=labels_nl[cls], edgecolors='k', alpha=0.8, s=60)

axes[1].set_xlabel('Petal Length (standardized)', fontsize=12)
axes[1].set_ylabel('Petal Width (standardized)', fontsize=12)
axes[1].set_title('Nonlinear LR (Feature Expansion)', fontsize=13)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.suptitle('Task 3B: Linear vs Nonlinear Decision Boundary', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Loss comparison: Linear vs Nonlinear LR on Task 3
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(range(1, len(lr_t3.loss_history) + 1), lr_t3.loss_history, 
        color='#e74c3c', linewidth=2, label='Linear LR')
ax.plot(range(1, len(lr_t3_expanded.loss_history) + 1), lr_t3_expanded.loss_history, 
        color='#2ecc71', linewidth=2, label='Nonlinear LR (expanded features)')
ax.set_xlabel('Iteration', fontsize=12)
ax.set_ylabel('Loss', fontsize=12)
ax.set_title('Loss Comparison: Linear vs Nonlinear LR (Task 3)', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Observations – Task 3 (Non-Linearly Separable Data)

* **Class Separation (Task 2 vs Task 3):** In Task 2, Setosa and Versicolor are clearly separated with a wide margin in the petal length–petal width space. In Task 3, Versicolor and Virginica overlap significantly — their petal dimensions share a region of ambiguity, making the problem inherently harder for linear classifiers.

* **Perceptron Behaviour on Non-Linearly Separable Data:** The Perceptron runs for all 100 epochs without converging (the misclassification count never reaches zero). This is expected — the Perceptron convergence theorem only guarantees convergence for linearly separable data. On non-separable data, the weight vector oscillates as the algorithm keeps updating for misclassified points that cannot all be correctly classified by any linear boundary.

* **Logistic Regression (Linear):** Despite the overlap, Logistic Regression still converges to a stable solution by minimizing the negative log-likelihood. Its training and test accuracy are lower than in Task 2, reflecting the inherent difficulty of the problem. The sigmoid function naturally handles overlapping classes by assigning probabilistic predictions rather than hard boundaries.

* **Nonlinear Feature Expansion:** By mapping the original 2D features to a 5D space φ(x) = [x₁, x₂, x₁x₂, x₁², x₂²], the Logistic Regression model can learn a curved (nonlinear) decision boundary in the original feature space. This typically improves accuracy on non-linearly separable data because the boundary can better conform to the true class distribution. The contour plot shows a curved boundary that better separates the two classes compared to the straight line of linear LR.

* **Test Accuracy Before vs After Expansion:** The test accuracy with feature expansion is generally equal to or higher than the linear version, demonstrating that the additional polynomial features help capture the structure of the overlapping classes. However, the improvement depends on the specific train-test split and the degree of overlap.

---
## Task 4: Gradient Descent with Momentum

Use the linear Logistic Regression problem from Task 3. Compare ordinary Batch Gradient Descent (β=0) with Gradient Descent with Momentum (β=0.9).

**Update rules:**
- $v_0 = 0$
- $v_{t+1} = \beta v_t - \eta \nabla J(\theta_t)$
- $\theta_{t+1} = \theta_t + v_{t+1}$

In [ ]:
class LogisticRegressionMomentum:
    """
    Logistic Regression with Gradient Descent + Momentum from scratch.
    
    Update rules:
        v_{t+1} = β * v_t - η * ∇J(θ_t)
        θ_{t+1} = θ_t + v_{t+1}
    
    When β = 0, this reduces to ordinary Batch Gradient Descent.
    """
    
    def __init__(self, lr=0.05, beta=0.0, max_iter=200):
        self.lr = lr       # η (learning rate)
        self.beta = beta   # β (momentum coefficient)
        self.max_iter = max_iter
        self.theta = None
        self.loss_history = []
    
    def sigmoid(self, z):
        z = np.clip(z, -500, 500)
        return 1.0 / (1.0 + np.exp(-z))
    
    def compute_loss(self, X_aug, y):
        m = len(y)
        h = self.sigmoid(X_aug @ self.theta)
        h = np.clip(h, 1e-10, 1 - 1e-10)
        loss = -(1/m) * np.sum(y * np.log(h) + (1 - y) * np.log(1 - h))
        return loss
    
    def fit(self, X, y):
        n_samples, n_features = X.shape
        X_aug = np.hstack([np.ones((n_samples, 1)), X])
        
        # Initialize θ₀ = 0, v₀ = 0
        self.theta = np.zeros(n_features + 1)
        v = np.zeros(n_features + 1)
        self.loss_history = []
        
        for t in range(self.max_iter):
            # Compute gradient
            h = self.sigmoid(X_aug @ self.theta)
            gradient = (1/n_samples) * (X_aug.T @ (h - y))
            
            # Momentum update
            v = self.beta * v - self.lr * gradient
            self.theta = self.theta + v
            
            loss = self.compute_loss(X_aug, y)
            self.loss_history.append(loss)
        
        return self
    
    def predict_proba(self, X):
        X_aug = np.hstack([np.ones((X.shape[0], 1)), X])
        return self.sigmoid(X_aug @ self.theta)
    
    def predict(self, X):
        return (self.predict_proba(X) >= 0.5).astype(int)
    
    def accuracy(self, X, y):
        return np.mean(self.predict(X) == y) * 100

In [ ]:
# Train with β = 0 (ordinary Batch Gradient Descent)
model_beta0 = LogisticRegressionMomentum(lr=0.05, beta=0.0, max_iter=200)
model_beta0.fit(X_train_t3_s, y_train_t3)

# Train with β = 0.9 (Gradient Descent with Momentum)
model_beta09 = LogisticRegressionMomentum(lr=0.05, beta=0.9, max_iter=200)
model_beta09.fit(X_train_t3_s, y_train_t3)

print("="*65)
print("GRADIENT DESCENT WITH MOMENTUM – RESULTS")
print("="*65)

In [ ]:
# Report loss at iterations 10, 50, 100, 200
checkpoints = [10, 50, 100, 200]

print(f"\n{'Iteration':<12} {'Loss (β=0)':<20} {'Loss (β=0.9)':<20}")
print("-" * 52)
for cp in checkpoints:
    loss_0 = model_beta0.loss_history[cp - 1]
    loss_09 = model_beta09.loss_history[cp - 1]
    print(f"{cp:<12} {loss_0:<20.6f} {loss_09:<20.6f}")

# Final training and test accuracy
print(f"\n{'Metric':<25} {'β=0':<15} {'β=0.9':<15}")
print("-" * 55)
print(f"{'Training Accuracy':<25} {model_beta0.accuracy(X_train_t3_s, y_train_t3):<15.2f} {model_beta09.accuracy(X_train_t3_s, y_train_t3):<15.2f}")
print(f"{'Test Accuracy':<25} {model_beta0.accuracy(X_test_t3_s, y_test_t3):<15.2f} {model_beta09.accuracy(X_test_t3_s, y_test_t3):<15.2f}")
print(f"{'Final Loss':<25} {model_beta0.loss_history[-1]:<15.6f} {model_beta09.loss_history[-1]:<15.6f}")

In [ ]:
# Plot loss versus iteration for both β values
fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(range(1, 201), model_beta0.loss_history, 
        color='#e74c3c', linewidth=2.5, label='β = 0 (Ordinary GD)')
ax.plot(range(1, 201), model_beta09.loss_history, 
        color='#3498db', linewidth=2.5, label='β = 0.9 (GD with Momentum)')

# Mark checkpoint iterations
for cp in checkpoints:
    ax.axvline(x=cp, color='gray', linestyle=':', alpha=0.4)
    ax.annotate(f'iter {cp}', xy=(cp, ax.get_ylim()[1]*0.95), fontsize=8, 
                color='gray', ha='center')

ax.set_xlabel('Iteration', fontsize=13)
ax.set_ylabel('Loss (Negative Log-Likelihood)', fontsize=13)
ax.set_title('Task 4: Loss vs Iteration — Ordinary GD vs GD with Momentum\n(η = 0.05, 200 iterations)', fontsize=14)
ax.legend(fontsize=12, loc='upper right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Zoomed-in view of loss for early iterations
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# First 50 iterations
axes[0].plot(range(1, 51), model_beta0.loss_history[:50], 
             color='#e74c3c', linewidth=2, label='β = 0')
axes[0].plot(range(1, 51), model_beta09.loss_history[:50], 
             color='#3498db', linewidth=2, label='β = 0.9')
axes[0].set_xlabel('Iteration', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].set_title('Early Convergence (Iterations 1–50)', fontsize=13)
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# Last 100 iterations
axes[1].plot(range(101, 201), model_beta0.loss_history[100:], 
             color='#e74c3c', linewidth=2, label='β = 0')
axes[1].plot(range(101, 201), model_beta09.loss_history[100:], 
             color='#3498db', linewidth=2, label='β = 0.9')
axes[1].set_xlabel('Iteration', fontsize=12)
axes[1].set_ylabel('Loss', fontsize=12)
axes[1].set_title('Late Convergence (Iterations 101–200)', fontsize=13)
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.suptitle('Task 4: Convergence Analysis – Early vs Late Iterations', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Decision boundary comparison for β=0 vs β=0.9
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for idx, (model, beta_val) in enumerate([(model_beta0, 0.0), (model_beta09, 0.9)]):
    ax = axes[idx]
    theta = model.theta
    
    colors = ['#3498db', '#e67e22']
    for cls in [0, 1]:
        mask = y_train_t3 == cls
        ax.scatter(X_train_t3_s[mask, 0], X_train_t3_s[mask, 1], c=colors[cls], 
                   label=['Versicolor', 'Virginica'][cls], edgecolors='k', alpha=0.8, s=60)
    
    x1_range = np.linspace(X_train_t3_s[:, 0].min() - 0.5, X_train_t3_s[:, 0].max() + 0.5, 200)
    if abs(theta[2]) > 1e-10:
        x2_boundary = -(theta[0] + theta[1] * x1_range) / theta[2]
        ax.plot(x1_range, x2_boundary, 'k--', linewidth=2, label='Decision Boundary')
    
    ax.set_xlabel('Petal Length (standardized)', fontsize=12)
    ax.set_ylabel('Petal Width (standardized)', fontsize=12)
    ax.set_title(f'β = {beta_val} (η = 0.05, 200 iter)', fontsize=13)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)

plt.suptitle('Task 4: Decision Boundary – Ordinary GD vs GD with Momentum', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### Observations – Task 4 (Gradient Descent with Momentum)

* **Convergence Behaviour:** Gradient Descent with Momentum (β = 0.9) converges noticeably faster than ordinary Gradient Descent (β = 0). At the same learning rate η = 0.05, the momentum-based optimizer reaches lower loss values in fewer iterations. This is clearly visible in the loss-vs-iteration plot.

* **Loss at Checkpoints:** At iterations 10, 50, 100, and 200, the loss for β = 0.9 is consistently lower than for β = 0. The momentum term accelerates the optimizer through flat regions and helps it maintain progress in the correct direction.

* **Final Accuracy:** Both methods ultimately reach similar training and test accuracy because the problem has a unique optimum and both converge given enough iterations. The key difference is the *speed* of convergence, not the final result.

* **Roles of $v_t$, $\beta$, and $\eta$:**
  - **$v_t$ (velocity):** Accumulates the exponentially weighted average of past gradients. It acts as a "memory" of previous gradient directions, allowing the optimizer to build up speed in consistent directions and dampen oscillations in inconsistent directions.
  - **$\beta$ (momentum coefficient):** Controls how much of the previous velocity is retained. β = 0 gives ordinary GD (no memory). β = 0.9 means 90% of the previous velocity is carried forward. Higher β gives stronger momentum, which accelerates convergence in low-curvature directions but can cause overshooting if set too high.
  - **$\eta$ (learning rate):** Controls the step size of each gradient update. It determines how much the current gradient contributes to the velocity. Together with β, it controls the effective step size: momentum amplifies the effective learning rate in consistent gradient directions by a factor of approximately $\frac{1}{1-\beta}$.

### Conclusion

Gradient Descent with Momentum is a simple yet powerful enhancement over vanilla gradient descent. By introducing a velocity term $v_t$ that accumulates past gradients, momentum smooths out noisy gradient signals and accelerates convergence, especially in loss landscapes with high curvature in some directions and low curvature in others (elongated "valleys"). The momentum coefficient β = 0.9 is a widely used default that provides a good balance between acceleration and stability. The learning rate η must be tuned carefully — with momentum, a smaller η can still achieve fast convergence because the velocity amplifies the effective step size in consistent directions.

---
## Summary Table

| Task | Classifier | Training Acc | Test Acc | Key Finding |
|------|-----------|-------------|---------|-------------|
| 1 | Naive Bayes (Titanic) | — | Reported above | Sex is strongest predictor; Laplace smoothing prevents zero probabilities |
| 2A | Perceptron (Setosa vs Versicolor) | ~100% | ~100% | Converges quickly on linearly separable data |
| 2B | Logistic Regression (Setosa vs Versicolor) | ~100% | ~100% | Smooth loss convergence; probabilistic outputs |
| 3A | Perceptron (Versicolor vs Virginica) | <100% | <100% | Fails to converge; oscillates on non-separable data |
| 3A | Logistic Regression (Versicolor vs Virginica) | <100% | <100% | Stable but limited by linear boundary |
| 3B | LR + Feature Expansion | ≥ Linear LR | ≥ Linear LR | Nonlinear boundary improves classification |
| 4 | GD vs Momentum | Similar | Similar | Momentum (β=0.9) converges faster |